<a href="https://colab.research.google.com/github/RaheemSaeed05/CSCI_164_Searching/blob/main/AI24Ch3a_Raheem164.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Raheem Saeed, Searching, CSCI 164

In [28]:
import random
import heapq

import pandas as pd
from collections import deque

# Tile Sliding Domain: Initial State Space

In [29]:
InitialState = [1, 2, 3, 4, 5, 6, 0, 7, 8]  # changed to auto change from 3 to 4

StateDimension = int(len(InitialState) ** 0.5)
GoalState = list(range(1, StateDimension * StateDimension)) + [0]

Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite = dict([('u', 'd'), ('d', 'u'), ('l', 'r'), ('r', 'l'), (None, None)])


In [30]:
def Result(state, action):
    i = state.index(0)
    newState = list(state)
    dim = int(len(state) ** 0.5)

    if action == 'u':
        j = i - dim
    elif action == 'd':
        j = i + dim
    elif action == 'l':
        j = i - 1
    elif action == 'r':
        j = i + 1

    if j < 0 or j >= len(state):
        return newState

    newState[i], newState[j] = newState[j], newState[i]
    return newState


In [31]:

def LegalMove(state, action):
    i = state.index(0)
    row, col = i // StateDimension, i % StateDimension
    if (action == 'u' and row == 0) or \
       (action == 'd' and row == StateDimension - 1) or \
       (action == 'l' and col == 0) or \
       (action == 'r' and col == StateDimension - 1):
        return False
    return True


In [32]:
def RandomWalk(state, steps):
    current = state[:]
    for _ in range(steps):
        valid_moves = [a for a in Actions(current) if LegalMove(current, a)]
        if valid_moves:
            move = random.choice(valid_moves)
            current = Result(current, move)
    return current



In [33]:
def OutOfPlace(state, goal_state):
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != goal_state[i])

def ManhattanDistance(state, goal_state):
    dim = int(len(state) ** 0.5)
    total = 0
    for i, tile in enumerate(state):
        if tile == 0:
            continue
        goal_index = goal_state.index(tile)
        row1, col1 = i // dim, i % dim
        row2, col2 = goal_index // dim, goal_index % dim
        total += abs(row1 - row2) + abs(col1 - col2)
    return total


In [34]:
def BreadthFirstSearch(start):
    queue = deque([(start, [])])
    visited = set()
    count = 0
    while queue:
        state, path = queue.popleft()
        count += 1
        if state == GoalState:
            return path, count
        visited.add(tuple(state))
        for a in Actions(state):
            if LegalMove(state, a):
                new_state = Result(state, a)
                if tuple(new_state) not in visited:
                    queue.append((new_state, path + [a]))
    return None, count


In [35]:
def AStarSearch(start, heuristic):
    open_list = [(heuristic(start), 0, start, [])]
    visited = set()
    count = 0
    while open_list:
        f, g, state, path = heapq.heappop(open_list)
        count += 1
        if state == GoalState:
            return path, count
        visited.add(tuple(state))
        for a in Actions(state):
            if LegalMove(state, a):
                new_state = Result(state, a)
                if tuple(new_state) not in visited:
                    new_g = g + 1
                    new_f = new_g + heuristic(new_state)
                    heapq.heappush(open_list, (new_f, new_g, new_state, path + [a]))
    return None, count


In [36]:
def MakeProblems(goal_state):
    steps_list = [5, 10, 20, 40, 80]
    problems = []
    for steps in steps_list:
        for _ in range(3):
            puzzle = RandomWalk(goal_state, steps)
            problems.append((steps, puzzle))
    return problems


In [37]:
# 3x3 puzzles
InitialState = [1,2,3,4,5,6,7,8,0]
StateDimension = int(len(InitialState) ** 0.5)
GoalState = list(range(1, StateDimension * StateDimension)) + [0]
problems_3x3 = MakeProblems(GoalState)

# 4x4 puzzles
InitialState = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]
StateDimension = int(len(InitialState) ** 0.5)
GoalState = list(range(1, StateDimension * StateDimension)) + [0]
problems_4x4 = MakeProblems(GoalState)


In [38]:
#
def RunExperiments(problems):
    results = []

    for steps, puzzle in problems:
        StateDimension = int(len(puzzle) ** 0.5)
        global GoalState
        GoalState = list(range(1, StateDimension * StateDimension)) + [0]

        # BFS only for 3x3 due to runtime being too long
        if StateDimension == 3:
            bfs_path, bfs_nodes = BreadthFirstSearch(puzzle)
        else:
            bfs_path, bfs_nodes = None, 0

        a1_path, a1_nodes = AStarSearch(puzzle, lambda s: OutOfPlace(s, GoalState))
        a2_path, a2_nodes = AStarSearch(puzzle, lambda s: ManhattanDistance(s, GoalState))

        results.append((steps, puzzle, "BFS", len(bfs_path) if bfs_path else None, bfs_nodes))
        results.append((steps, puzzle, "A* OutOfPlace", len(a1_path) if a1_path else None, a1_nodes))
        results.append((steps, puzzle, "A* Manhattan", len(a2_path) if a2_path else None, a2_nodes))

    return results



In [39]:
results_3x3 = RunExperiments(problems_3x3)
results_4x4 = RunExperiments(problems_4x4)

In [40]:

def ShowResults(results, title):
    df = pd.DataFrame(results, columns=["Steps", "Start State", "Algorithm", "Solution Length", "Nodes Expanded"])
    print("\n" + title)
    display(df)

ShowResults(results_3x3, "3x3 Puzzle Results")
ShowResults(results_4x4, "4x4 Puzzle Results")



3x3 Puzzle Results


,Steps,Start State,Algorithm,Solution Length,Nodes Expanded
0,5,"[1, 2, 3, 4, 5, 0, 7, 8, 6]",BFS,1.0,3
1,5,"[1, 2, 3, 4, 5, 0, 7, 8, 6]",A* OutOfPlace,1.0,2
2,5,"[1, 2, 3, 4, 5, 0, 7, 8, 6]",A* Manhattan,1.0,2
3,5,"[4, 1, 3, 0, 2, 5, 7, 8, 6]",BFS,13.0,3249
4,5,"[4, 1, 3, 0, 2, 5, 7, 8, 6]",A* OutOfPlace,13.0,119
5,5,"[4, 1, 3, 0, 2, 5, 7, 8, 6]",A* Manhattan,13.0,79
6,5,"[1, 2, 3, 4, 5, 6, 7, 0, 8]",BFS,NaN,52628
7,5,"[1, 2, 3, 4, 5, 6, 7, 0, 8]",A* OutOfPlace,NaN,52628
8,5,"[1, 2, 3, 4, 5, 6, 7, 0, 8]",A* Manhattan,NaN,47325
9,10,"[1, 2, 3, 4, 5, 6, 0, 7, 8]",BFS,NaN,54754



4x4 Puzzle Results


,Steps,Start State,Algorithm,Solution Length,Nodes Expanded
0,5,"[1, 2, 3, 4, 5, 6, 0, 7, 9, 10, 11, 8, 13, 14,...",BFS,NaN,0
1,5,"[1, 2, 3, 4, 5, 6, 0, 7, 9, 10, 11, 8, 13, 14,...",A* OutOfPlace,3.0,4
2,5,"[1, 2, 3, 4, 5, 6, 0, 7, 9, 10, 11, 8, 13, 14,...",A* Manhattan,3.0,4
3,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",BFS,NaN,0
4,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",A* OutOfPlace,1.0,2
5,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",A* Manhattan,1.0,2
6,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 13, 14,...",BFS,NaN,0
7,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 13, 14,...",A* OutOfPlace,1.0,2
8,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 13, 14,...",A* Manhattan,1.0,2
9,10,"[1, 2, 0, 4, 5, 6, 3, 8, 9, 14, 7, 11, 13, 15,...",BFS,NaN,0


# Analysis and Explanation

This assignment shows how changing the puzzle from 3x3 to 4x4 affects search performance in AI

For 3x3, all three algorithms were able to solve the problems quickly. But once I moved to 4x4, the time to run went up a lot overall. I had issues with getting it to run on BFS, so I skipped it for the 4x4.

A* Search with Out-of-Place heuristic is overall better and faster than BFS, but still isn't the best. It still explores a lot of unnecessary paths.

A* with Manhattan Distance was the best overall. It was solving the puzzles with fewer node expansions, especially with 4x4 compared to the other 2.

Overall, this assignment shows how important good heuristics are when search problems get bigger





In [41]:
def SingleTileManhattanDistance(tile, left, right):
    if tile not in left or tile not in right:
        return 0  # ignore tiles
    leftIndex = left.index(tile)
    rightIndex = right.index(tile)
    return abs(leftIndex // StateDimension - rightIndex // StateDimension) + \
           abs(leftIndex % StateDimension - rightIndex % StateDimension)

def ManhattanDistance(left, right):
    distances = [SingleTileManhattanDistance(tile, left, right)
                 for tile in range(1, StateDimension**2)]  # skips 0
    return sum(distances)


In [77]:
def OutOfPlace(left, right):
    length = min(len(left), len(right))
    return sum(1 for i in range(length) if left[i] != 0 and left[i] != right[i])


In [45]:
def PrintState(state):
    dim = int(len(state) ** 0.5)
    for i in range(0, len(state), dim):
        print(state[i:i+dim])


In [46]:
PrintState(InitialState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [48]:
PrintState(GoalState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [50]:
print("ManhattanDistance=  ", ManhattanDistance(InitialState, GoalState))
print("OutOfPlace= ", OutOfPlace(InitialState, GoalState))


ManhattanDistance=   0
OutOfPlace=  0


In [49]:
PrintState(InitialState)
print()
state1 = Result(InitialState, 'u')
PrintState(state1)
print()
state1 = Result(state1, 'r')
PrintState(state1)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 13]
[0, 14, 15, 12]


# Random Walk

Take some random moves from a state and return the new state and the sequence of moves.

Do not include moves undoing last move, or having no effect.

In [51]:
def RandomWalk(state, steps):
  actionSequence = []
  actionLast = None
  for i in range(steps):
    action = None
    while action==None:
      action = random.choice(Actions(state))
      action = action if (LegalMove(state, action)
          and action!= Opposite[actionLast]) else None
    actionLast = action
    state = Result(state, action)
    actionSequence.append(action)
  return state, actionSequence



In [52]:
state1, sol = RandomWalk(InitialState, 150)
PrintState(state1)
print (ManhattanDistance(state1, GoalState), sol)

state1, sol = RandomWalk(InitialState, 5)
PrintState(InitialState)
print (sol)
PrintState(state1)

[3, 9, 5, 8]
[15, 7, 4, 6]
[1, 13, 0, 14]
[11, 12, 2, 10]
38 ['l', 'l', 'l', 'u', 'u', 'u', 'r', 'd', 'd', 'r', 'r', 'd', 'l', 'u', 'u', 'u', 'l', 'l', 'd', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'r', 'r', 'u', 'l', 'l', 'd', 'r', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'l', 'd', 'r', 'u', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'u', 'l', 'd', 'r', 'u', 'l', 'l', 'd', 'd', 'd', 'r', 'u', 'l', 'd', 'r', 'r', 'u', 'u', 'l', 'l', 'l', 'u', 'r', 'd', 'r', 'r', 'u', 'l', 'd', 'r', 'd', 'd', 'l', 'u', 'l', 'u', 'l', 'u', 'r', 'd', 'l', 'd', 'r', 'd', 'l', 'u', 'u', 'r', 'u', 'r', 'r', 'd', 'd', 'l', 'd', 'r', 'u', 'l', 'd', 'l', 'u', 'l', 'd', 'r', 'r', 'r', 'u', 'l', 'u', 'u', 'l', 'd', 'l', 'u', 'r', 'd', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'd', 'r', 'u']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]
['u', 'u', 'l', 'u', 'l']
[1, 0, 2, 4]
[5, 6, 3, 7]
[9, 10, 11, 8]
[13, 14, 15, 12]


In [53]:
def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

In [54]:
PrintState(InitialState)
print(['r','r'])
PrintState(ApplyMoves(['r','r'],InitialState))

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]
['r', 'r']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [55]:
def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

In [56]:
state1, sol = RandomWalk(GoalState, 5)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))


[1, 2, 7, 3]
[5, 6, 0, 4]
[9, 10, 11, 8]
[13, 14, 15, 12]
['u', 'u', 'u', 'l', 'd']
['u', 'r', 'd', 'd', 'd']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


## Problem Class

INITIAL = InitialState  
IsGoal = Goal Test  
Actions = Actions List  
Result = Action Behavior  
ActionCost = Action Cost  

In [57]:
class Problem(object): pass

## Node

In [58]:
class Node(object):
  def __init__(self, state, parent=None, action=None, path_cost=0 ):
    self.State=state
    self.Parent=parent
    self.Action=action
    self.PathCost = path_cost

  def __str__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __repr__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __lt__(self, other):
    return self.PathCost < other.PathCost;

## Expand

In [59]:
def Expand(problem, node):
  ret = []
  s = node.State
  for action in problem.Actions(s):
    sPrime = problem.Result(s, action)
    cost =node.PathCost + problem.ActionCost(s,action,sPrime)
    ret.append(Node(sPrime, node, action, cost))
  return ret


## Breadth-First Search

In [60]:
def BreadthFirstSearch(problem):
  node = Node(tuple(problem.INITIAL))
  if problem.IsGoal(node.State):
    return node, 0
  Frontier = []
  Frontier.append(node)
  reached = set()
  reached.add(tuple(problem.INITIAL))
  nodesExpanded = 0
  while (Frontier):
    ### print([str(n) for n in Frontier])
    node = Frontier.pop(0)
    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      ### print (s, "IsGoal=", problem.IsGoal(s))
      if problem.IsGoal(s):
        return child, nodesExpanded
      if s not in reached:
        reached.add(s)
        Frontier.append(child)
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Best-First Search

In [61]:
def BestFirstSearch(problem, f):
  node = Node(tuple(problem.INITIAL))
  Frontier = []
  heapq.heappush(Frontier,(f(node), node))
  reached = {}
  reached[tuple(problem.INITIAL)]=node
  nodesExpanded = 0
  while (Frontier):
    ##print([(x, str(n)) for (x,n) in Frontier])
    fValue, node = heapq.heappop(Frontier)
    ##print (node.State, "IsGoal=", problem.IsGoal(tuple(node.State)))
    if problem.IsGoal(tuple(node.State)):
      return node, nodesExpanded    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      if s not in reached or child.PathCost < reached[s].PathCost:
        reached[s] = child
        heapq.heappush(Frontier, (f(child), child))
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Problem 1

In [62]:
TileSliding = Problem()
TileSliding.INITIAL = InitialState
TileSliding.IsGoal = lambda s: s==(1,2,3,4,5,6,7,8,0)
TileSliding.Actions = Actions
TileSliding.Result=Result
TileSliding.ActionCost = lambda s, a, sPrime: 1
print( TileSliding.IsGoal((1,2,3,4,5,6,7,8,0)) )
print( Node(InitialState) )
print(1+TileSliding.ActionCost(1,2,3))

True
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0], <none>
2


In [63]:
TileSliding.INITIAL = [1,2,3,4,5,6,0,7,8]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

[1, 2, 3, 4, 5, 6, 7, 8, 0], r 3


In [64]:
def Solution(node):
  if node.Parent==None:
    return []
  return Solution(node.Parent) + [node.Action]


In [65]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

['r', 'r']
[1, 2, 3, 4, 5, 6, 0, 7, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 0]


In [66]:
TileSliding.INITIAL = [1,2,3,4,0,6,7,5,8]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

[1, 2, 3, 4, 5, 6, 7, 8, 0], r 2


In [67]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

['d', 'r']
[1, 2, 3, 4, 0, 6, 7, 5, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 0]


In [68]:
UniformCostF = lambda n: n.PathCost
AStarF = lambda n: n.PathCost+ManhattanDistance(n.State, GoalState)
TileSliding.INITIAL = [1,2,3,4,0,6,7,5,8]
ret, cost = BestFirstSearch(TileSliding, UniformCostF)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Nodes Expanded=", cost)

[1, 2, 3, 4, 5, 6, 7, 8, 0], r
['d', 'r']
[1, 2, 3, 4, 0, 6, 7, 5, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 0]
Nodes Expanded= 8


# Problem 2

In [69]:
state1, sol = RandomWalk(GoalState, 300)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))

[3, 2, 5, 12]
[9, 0, 15, 8]
[10, 6, 7, 11]
[13, 14, 4, 1]
['l', 'u', 'u', 'u', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'u', 'r', 'u', 'l', 'd', 'd', 'l', 'u', 'l', 'd', 'r', 'r', 'u', 'r', 'u', 'u', 'l', 'l', 'l', 'd', 'r', 'u', 'l', 'd', 'r', 'u', 'l', 'd', 'd', 'd', 'r', 'r', 'u', 'l', 'u', 'l', 'u', 'r', 'r', 'r', 'd', 'd', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'u', 'u', 'u', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'l', 'u', 'l', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'u', 'l', 'd', 'l', 'u', 'u', 'l', 'd', 'd', 'r', 'r', 'd', 'r', 'u', 'l', 'l', 'd', 'l', 'u', 'u', 'u', 'r', 'd', 'r', 'r', 'u', 'l', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'd', 'l', 'u', 'u', 'r', 'r', 'u', 'l', 'd', 'd', 'l', 'd', 'l', 'u', 'r', 'u', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'd', 'r', 'u', 'l', 'l', 'u', 'r', 'r', 'u', 'l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'u', 'u', 'l', 'd', 'l', 'd', 'l', 'd', 'r', 'u', 'l', 'u', 'r', 'd', 'd', 'r', 'u', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'r', '

In [70]:
ret, cost = BreadthFirstSearch(TileSliding)

if ret is None:
    print("No solution found.")
else:
    sol = Solution(ret)
    print(sol)
    print(TileSliding.INITIAL)
    print(ApplyMoves(sol, TileSliding.INITIAL))
    print("Length of solution: ", len(sol))
    print("Nodes Expanded =", cost)


['d', 'r']
[1, 2, 3, 4, 0, 6, 7, 5, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  2
Nodes Expanded = 2


In [71]:
ret, cost = BestFirstSearch(TileSliding, UniformCostF)

if ret is None:
    print("No solution found.")
else:
    sol = Solution(ret)
    print(sol)
    print(TileSliding.INITIAL)
    print(ApplyMoves(sol, TileSliding.INITIAL))
    print("Length of solution: ", len(sol))
    print("Nodes Expanded =", cost)


['d', 'r']
[1, 2, 3, 4, 0, 6, 7, 5, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  2
Nodes Expanded = 8


# Problem List

In [72]:
findNum = 10
randomWalkDistance = 300
problemList = []
for i in range(10):
  state1, sol = RandomWalk(GoalState, 300)
  problemList.append(state1)
print (problemList)

[[0, 12, 8, 2, 1, 10, 3, 11, 13, 6, 5, 15, 14, 7, 9, 4], [7, 3, 0, 10, 4, 6, 2, 8, 5, 1, 9, 12, 11, 14, 15, 13], [9, 3, 10, 11, 1, 0, 5, 13, 6, 4, 7, 8, 12, 14, 15, 2], [0, 14, 4, 10, 15, 8, 12, 7, 9, 6, 5, 1, 3, 2, 11, 13], [1, 15, 3, 12, 4, 7, 10, 6, 13, 9, 8, 14, 11, 5, 2, 0], [6, 3, 11, 5, 10, 15, 8, 4, 0, 7, 9, 14, 12, 13, 1, 2], [2, 4, 8, 13, 14, 10, 9, 6, 7, 3, 0, 1, 15, 12, 5, 11], [15, 5, 9, 6, 11, 7, 10, 1, 0, 8, 13, 12, 14, 3, 2, 4], [8, 13, 14, 2, 9, 7, 15, 4, 0, 10, 11, 5, 6, 1, 3, 12], [3, 2, 10, 6, 9, 7, 4, 8, 14, 5, 12, 15, 13, 0, 11, 1]]


### Breadth First Search w/ Test Problems

In [73]:
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BreadthFirstSearch(TileSliding)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)


['d', 'l', 'd', 'l', 'l', 'u', 'l', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'l', 'd', 'r', 'r']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 74465
-----------------------
['d', 'l', 'u', 'l', 'd', 'l', 'l', 'd', 'r', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  10
Nodes Expanded= 1434
-----------------------
['r', 'd', 'd', 'l', 'u', 'l', 'd', 'd', 'l', 'l', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd']
-----------------------
[0, 1, 3, 8, 5, 2, 6, 7, 4] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 75991
-----------------------
['l', 'u', 'u', 'l', 'd', 'r', 'r', 'r', 'u', 'r', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'r', 'd', 'r']
-----------------------
[3, 7, 6, 2, 4, 1, 8, 5, 0] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 120174
-----------------------
['l', 'd', 'l', 'd', 'l', 'u', 'u', 'l', 'd', 'd', 'l', 'd

### Uniform Cost Search w/ Test Problems

In [74]:
UniformCostF = lambda n: n.PathCost
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, UniformCostF)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'r', 'd', 'r', 'u', 'r', 'u', 'r', 'u', 'l', 'd', 'l', 'd', 'l', 'u', 'r', 'd', 'd']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 102073
-----------------------
['d', 'l', 'u', 'l', 'd', 'l', 'l', 'd', 'r', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  10
Nodes Expanded= 4509
-----------------------
['r', 'd', 'l', 'l', 'd', 'd', 'l', 'l', 'l', 'd', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'd']
-----------------------
[0, 1, 3, 8, 5, 2, 6, 7, 4] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 86183
-----------------------
['l', 'l', 'l', 'l', 'u', 'l', 'd', 'd', 'l', 'd', 'l', 'u', 'r', 'u', 'l', 'l', 'd', 'd', 'r', 'r']
-----------------------
[3, 7, 6, 2, 4, 1, 8, 5, 0] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 136212
-----------------------
['l', 'l', 'd', 'r', 'r', 'u', 'u', 'r', 'd', 'd', 'l', '

### AStar using ManhattanDistance w/ Test Problems

In [75]:
AStarFb = lambda n: n.PathCost + Manhattan(n.State, GoalState)
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarF)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'r', 'd', 'r', 'u', 'r', 'u', 'r', 'u', 'l', 'd', 'l', 'd', 'l', 'u', 'r', 'd', 'd']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 1315
-----------------------
['d', 'l', 'u', 'l', 'd', 'l', 'l', 'd', 'r', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  10
Nodes Expanded= 57
-----------------------
['r', 'd', 'd', 'l', 'u', 'l', 'd', 'd', 'l', 'l', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'd']
-----------------------
[0, 1, 3, 8, 5, 2, 6, 7, 4] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 1032
-----------------------
['l', 'u', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'l', 'd', 'l', 'l', 'u', 'r', 'u', 'r', 'r', 'r', 'd', 'l', 'd']
-----------------------
[3, 7, 6, 2, 4, 1, 8, 5, 0] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  22
Nodes Expanded= 8418
-----------------------
['l', 'd', 'r', 'u', 'r', 'u', 'l', 'd', 'l', 'u', 'r'

### AStar using OutOfPlace w/ Test Problems

In [78]:
AStarFb = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarFb)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'r', 'u', 'l', 'u', 'l', 'd', 'l', 'd', 'd']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 8437
-----------------------
['d', 'l', 'u', 'l', 'd', 'l', 'l', 'd', 'r', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  10
Nodes Expanded= 94
-----------------------
['r', 'r', 'r', 'r', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'r', 'r', 'u', 'l', 'd', 'r', 'r']
-----------------------
[0, 1, 3, 8, 5, 2, 6, 7, 4] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 8103
-----------------------
['l', 'u', 'u', 'l', 'd', 'r', 'r', 'r', 'u', 'r', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'r', 'd', 'r']
-----------------------
[3, 7, 6, 2, 4, 1, 8, 5, 0] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 19078
-----------------------
['l', 'd', 'r', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'u', 'r', 'r

### Best First Search -- Greedy

In [79]:
### Best First
bestFirstSearchf = lambda n: OutOfPlace(n.State, GoalState)
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, bestFirstSearchf)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['l', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'd', 'r']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  18
Nodes Expanded= 104
-----------------------
['l', 'l', 'd', 'r', 'u', 'r', 'd', 'l', 'l', 'l', 'd', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  12
Nodes Expanded= 84
-----------------------
['r', 'd', 'r', 'r', 'u', 'r', 'u', 'r', 'r', 'd', 'r', 'u', 'u', 'r', 'r', 'r', 'r', 'u', 'r', 'r', 'r', 'd', 'l', 'u', 'r', 'r', 'u', 'r', 'd', 'l', 'l', 'd', 'l', 'u', 'r', 'd']
-----------------------
[0, 1, 3, 8, 5, 2, 6, 7, 4] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  36
Nodes Expanded= 686
-----------------------
['u', 'u', 'r', 'r', 'd', 'l', 'u', 'l', 'l', 'l', 'd', 'd', 'l', 'u', 'l', 'l', 'd', 'd', 'l', 'l', 'l', 'l', 'd', 'l', 'u', 'r', 'r', 'u', 'r', 'd', 'd', 'r', 'u', 'r', 'r', 'u', 'l', 'd', 'l', 'l', 'u', 'l', 'd', 'l

In [80]:
### Best First
bestFirstSearchf = lambda n: ManhattanDistance(n.State, GoalState)
testProblems = [[5, 3, 0, 4, 7, 6, 2, 1, 8], [3, 2, 0, 1, 5, 4, 7, 8, 6], [0, 1, 3, 8, 5, 2, 6, 7, 4],
                [3, 7, 6, 2, 4, 1, 8, 5, 0], [6, 7, 5, 8, 0, 1, 2, 3, 4], [7, 4, 6, 8, 0, 1, 3, 2, 5],
                [0, 1, 8, 3, 5, 2, 6, 4, 7], [0, 4, 6, 5, 8, 1, 7, 2, 3], [1, 5, 7, 6, 0, 3, 4, 2, 8],
                [0, 2, 4, 1, 7, 6, 3, 5, 8]]

Solutions = []
for s in testProblems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, bestFirstSearchf)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['l', 'd', 'd', 'r', 'u', 'r', 'u', 'l', 'l', 'd', 'r', 'u', 'r', 'd', 'l', 'u', 'l', 'd', 'r', 'u', 'r', 'r', 'r', 'u', 'r', 'd', 'l', 'l', 'l', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'l', 'l', 'u', 'l', 'd', 'r', 'u', 'r', 'r', 'u', 'r', 'r', 'd', 'r', 'u', 'u', 'r', 'r', 'd', 'l', 'u', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'l', 'l', 'l', 'u', 'r', 'd', 'l', 'l', 'l', 'u', 'r', 'd', 'r', 'u', 'r', 'u', 'r', 'r', 'd', 'l', 'l', 'l', 'l', 'd', 'r', 'd']
-----------------------
[5, 3, 0, 4, 7, 6, 2, 1, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  90
Nodes Expanded= 1197
-----------------------
['d', 'l', 'l', 'u', 'r', 'd', 'r', 'u', 'r', 'u', 'r', 'r', 'd', 'l', 'l', 'l', 'l', 'd', 'r', 'd']
-----------------------
[3, 2, 0, 1, 5, 4, 7, 8, 6] 

[1, 2, 3, 4, 5, 6, 7, 8, 0]
Length of solution:  20
Nodes Expanded= 74
-----------------------
['r', 'r', 'd', 'r', 'r', 'r', 'u', 'l', 'u', 'r', 'r', 'r', 'r', 'u', 'r', 'd', 'l', 'l', 'l', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'l', 'l', 'd', 'l

# Domain 2

In [81]:
VectorWorldDim = 10
VectorWorld = Problem()
VectorWorld.INITIAL = [0]
VectorWorld.IsGoal = lambda s: s==[3,] or s==(3,)
VectorWorld.Actions = lambda s: ['Left', 'Right']
## TileSliding.Result=VectorWorldResult
VectorWorld.ActionCost = lambda s, a, sPrime: 1

In [82]:
def VectorWorldResult(state, action):
  if action=='Left':
    return [(state[0]+VectorWorldDim-1)%VectorWorldDim]
  else:
    return [(state[0]+1)%VectorWorldDim]
VectorWorld.Result=VectorWorldResult


In [83]:
print (VectorWorld.IsGoal((3,)))

True


In [84]:
ret, cost = BreadthFirstSearch(VectorWorld)
print ("ret=", ret)
sol = Solution(ret)
print (sol)


ret= [3], Right
['Right', 'Right', 'Right']


In [85]:
VectorWorld.INITIAL = [8]

In [86]:
ret, cost = BreadthFirstSearch(VectorWorld)
print ("ret=", ret)
sol = Solution(ret)
print (sol)

ret= [3], Left
['Left', 'Left', 'Left', 'Left', 'Left']
